<a href="https://colab.research.google.com/github/Sr-PiCINiNi/Bot-Whatsapp/blob/main/Uniring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
import sys, torch
print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:", torch.cuda.is_available())


Wed Jun 24 03:47:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pathlib
!git clone https://github.com/VAST-AI-Research/UniRig
%cd UniRig
# instala o requirements SEM flash_attn (deixamos pro fim, só se precisar)
req = [l for l in pathlib.Path("requirements.txt").read_text().splitlines()
       if l.strip() and "flash" not in l.lower()]
pathlib.Path("requirements_nofa.txt").write_text("\n".join(req))
!pip install -q -r requirements_nofa.txt
!pip install -q numpy==1.26.4
print("OK deps base")


Cloning into 'UniRig'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 376 (delta 104), reused 84 (delta 84), pack-reused 230 (from 1)
Receiving objects: 100% (376/376), 21.84 MiB | 15.43 MiB/s, done.
Resolving deltas: 100% (168/168), done.
Filtering content: 100% (4/4), 53.60 MiB | 4.42 MiB/s, done.
/content/UniRig
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement bpy==4.2 (from versions: none)
ERROR: No matching distribution found for bpy==4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 req

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [ ]:
import os, pathlib
if not os.path.isdir("/content/UniRig"):
    !git clone https://github.com/VAST-AI-Research/UniRig
%cd /content/UniRig

!conda create -n unirig python=3.11 -y
R = "conda run -n unirig"   # roda tudo no ambiente certo

# torch 2.4 + cu121 (tem wheel pronto pra todas as deps; roda na T4 numa boa)
!{R} pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

# requirements SEM flash_attn (instalamos só se precisar); bpy==4.2 já funciona no 3.11
req = [l for l in pathlib.Path("requirements.txt").read_text().splitlines()
       if l.strip() and "flash" not in l.lower()]
pathlib.Path("requirements_nofa.txt").write_text("\n".join(req))
!{R} pip install -q -r requirements_nofa.txt
!{R} pip install -q numpy==1.26.4 spconv-cu120
!{R} pip install -q torch_scatter torch_cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html --no-cache-dir
print("OK ambiente 'unirig' pronto")


/content/UniRig
Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/unirig

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB

In [ ]:
from google.colab import files
up = files.upload()
glb = list(up.keys())[0]
print("arquivo:", glb)


Saving Meshy_AI_Verdant_Dragon_0624032545_texture.fbx to Meshy_AI_Verdant_Dragon_0624032545_texture.fbx
arquivo: Meshy_AI_Verdant_Dragon_0624032545_texture.fbx


In [ ]:
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
print("pronto -> results/rigged.glb")


Info: Deleted 1 data-block(s)
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
FBX version: 7400
save to: tmp/Meshy_AI_Verdant_Dragon_0624032545_texture
1 models processed
done
done


100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
/content/UniRig/src/model/michelangelo/models/modules/checkpoint.py:65: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd()
/content/UniRig/src/model/michelangelo/models/modules/checkpoint.py:73: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(bwd=bwd)
/usr/local/envs/unirig/lib/python3.11/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(a

In [ ]:
from google.colab import files
files.download("results/rigged.glb")


FileNotFoundError: Cannot find file: results/rigged.glb

In [ ]:
!conda run -n unirig pip install -q https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3.post1/flash_attn-2.8.3.post1+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
print("flash_attn instalado")


flash_attn instalado


In [ ]:
%cd /content/UniRig
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
print("pronto -> results/rigged.glb")


/content/UniRig
0 models processed
done
load task config: configs/task/quick_inference_skeleton_articulationxl_ar_256.yaml
load data config: configs/data/quick_inference.yaml
load transform config: configs/transform/inference_ar_transform.yaml
load tokenizer config: configs/tokenizer/tokenizer_parts_articulationxl_256.yaml
load model config: configs/model/unirig_ar_350m_1024_81920_float32.yaml
load system config: configs/system/ar_inference_articulationxl.yaml

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]FlashAttention only supports Ampere GPUs or newer.

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]
done


0it [00:00, ?it/s]
0it [00:00, ?it/s]
/content/UniRig/src/model/michelangelo/models/modules/checkpoint.py:65: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
 

In [ ]:
%cd /content/UniRig
import re, pathlib

# 1) ESQUELETO (transformers): flash_attention_2 -> sdpa
chg = 0
for f in pathlib.Path(".").rglob("*"):
    if f.is_file() and f.suffix in (".py", ".yaml", ".yml", ".json"):
        t = f.read_text(errors="ignore")
        if "flash_attention_2" in t:
            f.write_text(t.replace("flash_attention_2", "sdpa"))
            print("sdpa  <-", f); chg += 1

# 2) PESOS (flash_attn MHA): força use_flash_attn=False em toda chamada MHA(...)
def add_noflash(m):
    inside = m.group(1)
    return m.group(0) if "use_flash_attn" in inside else f"MHA({inside}, use_flash_attn=False)"
for f in pathlib.Path("src").rglob("*.py"):
    t = f.read_text(errors="ignore")
    if re.search(r"\bMHA\(", t):
        t2 = re.sub(r"\bMHA\(([^()]*)\)", add_noflash, t)
        if t2 != t:
            f.write_text(t2); print("MHA no-flash <-", f)
print("OK -> flash_attention_2 trocado em", chg, "arquivo(s)")


/content/UniRig
sdpa  <- configs/model/unirig_ar_350m_1024_81920_float32.yaml
MHA no-flash <- src/model/unirig_skin.py
OK -> flash_attention_2 trocado em 1 arquivo(s)


In [ ]:
%cd /content/UniRig
!rm -rf results tmp
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
print("pronto -> results/rigged.glb")


/content/UniRig
Info: Deleted 1 data-block(s)
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
FBX version: 7400
save to: tmp/Meshy_AI_Verdant_Dragon_0624032545_texture
1 models processed
done
load task config: configs/task/quick_inference_skeleton_articulationxl_ar_256.yaml
load data config: configs/data/quick_inference.yaml
load transform config: configs/transform/inference_ar_transform.yaml
load tokenizer config: configs/tokenizer/tokenizer_parts_articulationxl_256.yaml
load model config: configs/model/unirig_ar_350m_1024_81920_float32.yaml
load system config: configs/system/ar_inference_articulationxl.yaml

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]
FBX export starting... 'results/skeleton.fbx'
export finished in 0.0427 sec.

Predicting DataLoader 0: 100%|██████████| 1/1 [00:26<00:00,  0.04it

In [ ]:
%cd /content/UniRig
import pathlib, re

# (a) ESQUELETO (transformers): flash_attention_2 -> sdpa
for f in pathlib.Path(".").rglob("*"):
    if f.is_file() and f.suffix in (".py", ".yaml", ".yml", ".json"):
        t = f.read_text(errors="ignore")
        if "flash_attention_2" in t:
            f.write_text(t.replace("flash_attention_2", "sdpa"))

# (b) PESOS - MHA: use_flash_attn=False em toda chamada MHA(...)
for f in pathlib.Path("src").rglob("*.py"):
    t = f.read_text(errors="ignore")
    if re.search(r"\bMHA\(", t):
        t2 = re.sub(r"\bMHA\(([^()]*)\)",
                    lambda m: m.group(0) if "use_flash_attn" in m.group(1)
                    else f"MHA({m.group(1)}, use_flash_attn=False)", t)
        if t2 != t: f.write_text(t2)

# (c) PESOS - PTv3 (Point Transformer): desliga o flash (usa atenção PyTorch)
p = pathlib.Path("src/model/pointcept/models/PTv3Object.py")
p.write_text(p.read_text().replace("self.enable_flash = enable_flash",
                                    "self.enable_flash = False"))

# confere
import subprocess
print("flash_attention_2 restante:",
      subprocess.run("grep -rl flash_attention_2 . | wc -l", shell=True, capture_output=True, text=True).stdout.strip())
print("PTv3 enable_flash=False:", "self.enable_flash = False" in p.read_text())
print(">>> PATCHES APLICADOS <<<")


/content/UniRig
flash_attention_2 restante: 0
PTv3 enable_flash=False: True
>>> PATCHES APLICADOS <<<


In [ ]:
%cd /content/UniRig
!rm -rf results tmp
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
import os
print("skeleton:", os.path.exists("results/skeleton.fbx"),
      "| skin:", os.path.exists("results/skin.fbx"),
      "| RIGGED:", os.path.exists("results/rigged.glb"))


/content/UniRig
Info: Deleted 1 data-block(s)
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
FBX version: 7400
save to: tmp/Meshy_AI_Verdant_Dragon_0624032545_texture
1 models processed
done
load task config: configs/task/quick_inference_skeleton_articulationxl_ar_256.yaml
load data config: configs/data/quick_inference.yaml
load transform config: configs/transform/inference_ar_transform.yaml
load tokenizer config: configs/tokenizer/tokenizer_parts_articulationxl_256.yaml
load model config: configs/model/unirig_ar_350m_1024_81920_float32.yaml
load system config: configs/system/ar_inference_articulationxl.yaml

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]
FBX export starting... 'results/skeleton.fbx'
export finished in 0.0445 sec.

Predicting DataLoader 0: 100%|██████████| 1/1 [00:25<00:00,  0.04it

In [ ]:
from google.colab import files
files.download("results/rigged.glb")


FileNotFoundError: Cannot find file: results/rigged.glb

In [ ]:
import pathlib

file_path = pathlib.Path('src/model/pointcept/models/PTv3Object.py')

if file_path.exists():
    current_content = file_path.read_text()

    # Apply the specific replacements to fix patch_size_max references
    # 1. Replace the problematic line in Block.forward
    modified_content = current_content.replace(
        "max_seqlen = offset2bincount(point.offset).min().tolist(), self.patch_size_max",
        "max_seqlen = offset2bincount(point.offset).min().tolist(), self.patch_size"
    )

    # 2. Remove patch_size_max parameter from PTv3Object.__init__ signature if it's still there
    # This assumes 'patch_size_max=8,' is within the __init__ signature line.
    modified_content = modified_content.replace("        patch_size_max=8,", "")

    # 3. Remove self.patch_size_max assignment in PTv3Object.__init__ if it's still there
    modified_content = modified_content.replace("        self.patch_size_max = patch_size_max", "")

    # 4. Remove the assertion involving patch_size_max if it's still there
    modified_content = modified_content.replace(
        "        assert self.patch_size_max == self.patch_size, \"patch_size_max must be equal to patch_size when enable_flash is False\"",
        ""
    )

    # Save the modified content back to the file
    file_path.write_text(modified_content)
    print("Re-applied patch_size_max fixes to src/model/pointcept/models/PTv3Object.py.")
    print(file_path.read_text())
else:
    print(f"File not found: {file_path}")

Modified src/model/pointcept/models/PTv3Object.py to fix patch_size_max error.
from functools import partial
from addict import Dict
import math
import torch
import torch.nn as nn
import spconv.pytorch as spconv
import torch_scatter
from timm.models.layers import DropPath
from typing import Union
from einops import rearrange

try:
    import flash_attn
except ImportError:
    flash_attn = None

from .utils.misc import offset2bincount
from .utils.structure import Point
from .modules import PointModule, PointSequential


class RPE(torch.nn.Module):
    def __init__(self, patch_size, num_heads):
        super().__init__()
        self.patch_size = patch_size
        self.num_heads = num_heads
        self.pos_bnd = int((4 * patch_size) ** (1 / 3) * 2)
        self.rpe_num = 2 * self.pos_bnd + 1
        self.rpe_table = torch.nn.Parameter(torch.zeros(3 * self.rpe_num, num_heads))
        torch.nn.init.trunc_normal_(self.rpe_table, std=0.02)

    def forward(self, coord):
        idx = (
  

In [ ]:
%cd /content/UniRig
import urllib.request
p = "src/model/pointcept/models/PTv3Object.py"
s = urllib.request.urlopen("https://raw.githubusercontent.com/VAST-AI-Research/UniRig/main/src/model/pointcept/models/PTv3Object.py").read().decode()
s = s.replace("self.enable_flash = enable_flash", "self.enable_flash = False")  # usa atenção PyTorch
s = s.replace("self.patch_size_max", "self.patch_size")                        # atributo que faltava
s = s.replace("self.attn_drop(attn)", "attn")                                  # attn_drop é float no init-flash; pula (dropout=identidade no inference)
open(p, "w").write(s)
print("enable_flash False:", "self.enable_flash = False" in s,
      "| patch_size_max sumiu:", "patch_size_max" not in s,
      "| attn_drop(attn) sumiu:", "self.attn_drop(attn)" not in s)


/content/UniRig
enable_flash False: True | patch_size_max sumiu: False | attn_drop(attn) sumiu: True


In [ ]:
%cd /content/UniRig
import urllib.request
p = "src/model/pointcept/models/PTv3Object.py"
s = urllib.request.urlopen("https://raw.githubusercontent.com/VAST-AI-Research/UniRig/main/src/model/pointcept/models/PTv3Object.py").read().decode()
s = s.replace("self.enable_flash = enable_flash", "self.enable_flash = False")  # usa atenção PyTorch
s = s.replace("self.patch_size_max", "self.patch_size")                        # atributo que faltava
s = s.replace("self.attn_drop(attn)", "attn")                                  # attn_drop é float no init-flash; pula (dropout=identidade no inference)
open(p, "w").write(s)
print("enable_flash False:", "self.enable_flash = False" in s,
      "| patch_size_max sumiu:", "patch_size_max" not in s,
      "| attn_drop(attn) sumiu:", "self.attn_drop(attn)" not in s)


/content/UniRig
enable_flash False: True | patch_size_max sumiu: False | attn_drop(attn) sumiu: True


In [ ]:
%cd /content/UniRig
!rm -rf results tmp
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
import os
print("skeleton:", os.path.exists("results/skeleton.fbx"),
      "| skin:", os.path.exists("results/skin.fbx"),
      "| RIGGED:", os.path.exists("results/rigged.glb"))


/content/UniRig
Info: Deleted 1 data-block(s)
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
FBX version: 7400
save to: tmp/Meshy_AI_Verdant_Dragon_0624032545_texture
1 models processed
done
load task config: configs/task/quick_inference_skeleton_articulationxl_ar_256.yaml
load data config: configs/data/quick_inference.yaml
load transform config: configs/transform/inference_ar_transform.yaml
load tokenizer config: configs/tokenizer/tokenizer_parts_articulationxl_256.yaml
load model config: configs/model/unirig_ar_350m_1024_81920_float32.yaml
load system config: configs/system/ar_inference_articulationxl.yaml

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]
FBX export starting... 'results/skeleton.fbx'
export finished in 0.0446 sec.

Predicting DataLoader 0: 100%|██████████| 1/1 [00:25<00:00,  0.04it

In [ ]:
%cd /content/UniRig
!rm -rf results tmp
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{glb}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{glb}" --output results/rigged.glb
import os
print("skeleton:", os.path.exists("results/skeleton.fbx"),
      "| skin:", os.path.exists("results/skin.fbx"),
      "| RIGGED:", os.path.exists("results/rigged.glb"))


/content/UniRig
Info: Deleted 1 data-block(s)
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
Now processing Meshy_AI_Verdant_Dragon_0624032545_texture.fbx...
FBX version: 7400
save to: tmp/Meshy_AI_Verdant_Dragon_0624032545_texture
1 models processed
done
load task config: configs/task/quick_inference_skeleton_articulationxl_ar_256.yaml
load data config: configs/data/quick_inference.yaml
load transform config: configs/transform/inference_ar_transform.yaml
load tokenizer config: configs/tokenizer/tokenizer_parts_articulationxl_256.yaml
load model config: configs/model/unirig_ar_350m_1024_81920_float32.yaml
load system config: configs/system/ar_inference_articulationxl.yaml

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]
FBX export starting... 'results/skeleton.fbx'
export finished in 0.0452 sec.

Predicting DataLoader 0: 100%|██████████| 1/1 [00:25<00:00,  0.04it

In [ ]:
from google.colab import drive
drive.mount('/content/drive')